# All Segmentation Runs — Master Dice Comparison

Computes and displays Dice scores for every run with local inference results.

| Group | Runs | Eval cohort |
|---|---|---|
| Fake PD only | run_002_v2 | D1 (10pt) |
| v5 | v5_realPD, v5_mixed | D1 (10pt) |
| v7 | v7_realPD, v7_mixed | D1 (10pt) |
| v8 (17pt) | v8_baseline, v8_pseudoPD, v8_realPD, v8_mixed | 17pt |


In [ ]:
import os, glob, re
import numpy as np
import nibabel as nib
import SimpleITK as sitk
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import zoom

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
BASE = "/Users/anshikabajpai/Desktop/github/RegGAN_domain_adaptation_v2"
RES  = f"{BASE}/results/final_results"
GT_DIR_D1  = "/Users/anshikabajpai/Desktop/AImed-lab/SEGMENTATIONS/PD-segmentations-final"
GT_DIR_NEW = "/Users/anshikabajpai/Downloads/final-segmentations"
PD_DIR     = "/Users/anshikabajpai/Desktop/AImed-lab/IU-Dess-dataset/iu-dataset/pd-files"

D1_PATIENTS = [
    "AC0D5A4D78B628","AC0D7BF72F7712","AC0D3459553205","AC14D3737C0482",
    "AC19E7C19827FF","AC149BC218E75C","AC111633B463BB","AC13300201B926",
    "AC12026D14291F","AC13637DA25399",
]
NEW_PATIENTS = [
    "AC000550763509","AC005135D3495B","AC04433E37DB66","AC056ADCE8BE28",
    "AC07607B9E5295","AC0D1A9818A6FC","AC0F11041F5180",
]

# All runs — name: (dir, label, eval_cohort)
RUNS = {
    "baseline_d1":  (f"{RES}/real_pd_predictions_baseline",    "Baseline (DESS)",          "D1"),
    "run_002_v2":   (f"{RES}/real_pd_predictions_run002_v2",   "Fake PD only (run_002_v2)","D1"),
    "v5_realPD":    (f"{RES}/real_pd_predictions_v5_mixed",    "v5 real PD only",          "D1"),  # check if separate dir exists
    "v5_mixed":     (f"{RES}/real_pd_predictions_v5_mixed",    "v5 Mixed (syn+real)",      "D1"),
    "v7_realPD":    (f"{RES}/real_pd_predictions_v7_realPD",   "v7 Real PD only",          "D1"),
    "v7_mixed":     (f"{RES}/real_pd_predictions_v7_mixed",    "v7 Mixed (syn+real)",      "D1"),
    "v8_baseline":  (f"{RES}/real_pd_predictions_v8_baseline", "v8 Baseline",              "17pt"),
    "v8_pseudoPD":  (f"{RES}/real_pd_predictions_v8_pseudoPD","v8 Pseudo PD (run_002_v2)","17pt"),
    "v8_realPD":    (f"{RES}/real_pd_predictions_v8_realPD",   "v8 Real PD only (v7_realPD ckpt)","17pt"),
    "v8_mixed":     (f"{RES}/real_pd_predictions_v8_mixed",    "v8 Mixed (v7_mixed ckpt)", "17pt"),
}
print("Runs defined:", list(RUNS.keys()))

In [ ]:
# ── Helper functions ───────────────────────────────────────────────────────
def get_pid(fname):
    m = re.match(r"(AC[A-F0-9]+)", os.path.basename(fname), re.IGNORECASE)
    return m.group(1) if m else None

def patients_in_dir(directory, pattern="*.nii.gz"):
    if not os.path.isdir(directory): return {}
    files = glob.glob(os.path.join(directory, pattern))
    return {get_pid(f): f for f in files if get_pid(f)}

def load_vol(path):
    return np.asarray(nib.load(path).get_fdata())

def load_nrrd_binary(path):
    img = sitk.ReadImage(path)
    arr = sitk.GetArrayFromImage(img)
    if arr.ndim == 4:
        arr = (arr.max(axis=0) > 0).astype(np.uint8)
        out = sitk.GetImageFromArray(arr)
        try:
            comp = sitk.VectorIndexSelectionCast(img, 0); out.CopyInformation(comp)
        except Exception:
            out.SetSpacing(img.GetSpacing()[:3]); out.SetOrigin(img.GetOrigin()[:3])
            out.SetDirection(img.GetDirection()[:9])
    else:
        arr = (arr > 0).astype(np.uint8)
        out = sitk.GetImageFromArray(arr); out.CopyInformation(img)
    try: out = sitk.DICOMOrient(out, "RAS")
    except Exception: pass
    sp = out.GetSpacing()
    arr = sitk.GetArrayFromImage(out).astype(np.float32).transpose(2,1,0)
    fa, fs = float(sp[1])/min(sp[1],sp[2]), float(sp[2])/min(sp[1],sp[2])
    if abs(fa-1.0)>0.02 or abs(fs-1.0)>0.02:
        arr = zoom(arr, (1.0, fa, fs), order=0)
    _, nA, nS = arr.shape
    if nA != 384 or nS != 384:
        arr = zoom(arr, (1.0, 384/nA, 384/nS), order=0)
    return (arr > 0.5).astype(bool)

def merge_meniscus(pred): return (pred > 0).astype(bool)

def dice_score(pred_bin, gt_bin):
    i = (pred_bin & gt_bin).sum(); d = pred_bin.sum() + gt_bin.sum()
    return float(2*i/d) if d > 0 else 1.0

def best_class_dice(pred_5class, gt_bin):
    best_d, best_c = -1.0, -1
    for c in range(1,5):
        d = dice_score((pred_5class==c), gt_bin)
        if d > best_d: best_d, best_c = d, c
    return best_d, best_c

In [ ]:
# ── Load GT masks ──────────────────────────────────────────────────────────
all_pids = D1_PATIENTS + NEW_PATIENTS
gt_masks = {}
for pid in all_pids:
    hits = glob.glob(os.path.join(GT_DIR_D1,  f"{pid}*.seg.nrrd")) or \
           glob.glob(os.path.join(GT_DIR_NEW, f"{pid}*.seg.nrrd"))
    if hits: gt_masks[pid] = hits[0]
    else: print(f"WARNING: No GT for {pid}")
print(f"GT masks: {len(gt_masks)}/17")

In [ ]:
# ── Compute Dice for all runs ──────────────────────────────────────────────
# Returns: {run_name: {pid: dice}}
run_results = {}

for run_name, (pred_dir, label, cohort) in RUNS.items():
    preds = patients_in_dir(pred_dir)
    if not preds:
        print(f"SKIP {run_name} — dir not found or empty: {pred_dir}")
        continue

    eval_pids = D1_PATIENTS if cohort == "D1" else all_pids
    run_results[run_name] = {}

    for pid in eval_pids:
        if pid not in gt_masks or pid not in preds:
            continue
        gt_bin = load_nrrd_binary(gt_masks[pid])
        pred_vol = load_vol(preds[pid])

        if run_name in ("baseline_d1", "v8_baseline"):
            d, _ = best_class_dice(pred_vol.astype(np.int16), gt_bin)
        else:
            d = dice_score(merge_meniscus(pred_vol), gt_bin)

        run_results[run_name][pid] = d

    vals = list(run_results[run_name].values())
    print(f"{run_name:<20} n={len(vals):2d}  mean={np.mean(vals):.4f}  "
          f"min={np.min(vals):.4f}  max={np.max(vals):.4f}")

In [ ]:
# ── Master summary table ───────────────────────────────────────────────────
rows = []
for run_name, (pred_dir, label, cohort) in RUNS.items():
    if run_name not in run_results: continue
    vals = list(run_results[run_name].values())
    rows.append({
        "Run":          run_name,
        "Label":        label,
        "Cohort":       cohort,
        "N patients":   len(vals),
        "Mean Dice":    round(np.mean(vals), 4) if vals else float("nan"),
        "Min Dice":     round(np.min(vals),  4) if vals else float("nan"),
        "Max Dice":     round(np.max(vals),  4) if vals else float("nan"),
    })

df_summary = pd.DataFrame(rows)
display(df_summary)
df_summary.to_csv(f"{BASE}/all_runs_dice_summary.csv", index=False)
print("Saved → all_runs_dice_summary.csv")

In [ ]:
# ── Per-patient Dice table (all runs × all patients) ───────────────────────
all_run_names = list(run_results.keys())
per_patient = []
for pid in all_pids:
    cohort = "D1" if pid in D1_PATIENTS else "new"
    row = {"Patient": pid, "Cohort": cohort}
    for rn in all_run_names:
        row[rn] = round(run_results[rn].get(pid, float("nan")), 4)
    per_patient.append(row)

df_perpatient = pd.DataFrame(per_patient)
display(df_perpatient)
df_perpatient.to_csv(f"{BASE}/all_runs_per_patient_dice.csv", index=False)
print("Saved → all_runs_per_patient_dice.csv")

In [ ]:
# ── Bar chart: mean Dice per run ───────────────────────────────────────────
colors_d1  = "#4C9BE8"
colors_17  = "#E87B4C"

fig, ax = plt.subplots(figsize=(16, 6))
run_labels, means, bar_colors = [], [], []
for run_name, (_, label, cohort) in RUNS.items():
    if run_name not in run_results: continue
    vals = list(run_results[run_name].values())
    run_labels.append(label)
    means.append(np.mean(vals) if vals else 0)
    bar_colors.append(colors_17 if cohort == "17pt" else colors_d1)

x = range(len(run_labels))
bars = ax.bar(x, means, color=bar_colors, alpha=0.85, edgecolor="white", linewidth=0.5)
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{m:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_xticks(list(x))
ax.set_xticklabels(run_labels, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Mean Dice Score", fontsize=12)
ax.set_ylim(0, 1.0)
ax.set_title("Mean Meniscus Dice — All Segmentation Runs", fontweight="bold", fontsize=13)
ax.grid(axis="y", alpha=0.3)

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=colors_d1, label="Eval: D1 (10pt)"),
                   Patch(color=colors_17,  label="Eval: 17pt")],
          loc="upper left", fontsize=10)
plt.tight_layout()
plt.savefig(f"{BASE}/all_runs_dice_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → all_runs_dice_summary.png")